In [1]:
from pydantic_ai import format_as_xml
from setup_quarto import setup_dot_env

ctx = setup_dot_env()

/Users/mattijnvanhoek/hkv/talk-with-fews/.pixi/envs/default/lib/python3.13/site-packages/pydantic/fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warn(


### XML Validation

In [2]:
# xml snippet on root level with direct parent with missing nested elements
# snippet should be based on pi-timeseries.xsd
xml_snippet = """
<series>
    <header>
        <type>instantaneous</type>
        
        <parameterId>WaterLevel</parameterId>
        <timeStep unit="second" multiplier="3600"/>
        <startDate date="2006-08-23" time="17:00:00"/>
        <endDate date="2006-08-24" time="00:00:00"/>
        <missVal>-8888.0</missVal>
    </header>
    <event date="2006-08-23" time="17:00:00" value="8.66"/>
</series>
"""

In [3]:
from talk_with_fews.tools.validator.validate_xml import validate_xml
from talk_with_fews.models.validate_xml import XMLValidationOutput, XMLValidationInput
from talk_with_fews.models.fews_schema_options import FEWSSchemaOption

xml_validation_input = XMLValidationInput(
    xml_content=xml_snippet, schema_option=FEWSSchemaOption.PI_TIMESERIES
)
xml_validation_output: XMLValidationOutput = await validate_xml(
    ctx=ctx, input=xml_validation_input
)

In [4]:
print(format_as_xml(xml_validation_output))

<is_valid>False</is_valid>
<errors>
  <item>
    <reason>Unexpected child with tag 'parameterId' at position 2. Tag 'locationId' expected.</reason>
    <xpath>Found direct parent 'TimeSeries' for 'series'</xpath>
  </item>
</errors>
<warnings />
<schema_location>data/fews_schemas/fewsdocs_deltares_nl/pi-schemas/pi_timeseries.xsd</schema_location>


In [5]:
# xml snippet with indirect parent to root with missing nested elements
xml_snippet = """
	<general>
		<exportType>SomeValidExportType</exportType>
		<folder>MyExportFolder</folder>
		<exportFileName>
			<prefix>
				<timeZeroFormattingString>yyyyMMddHHmm</timeZeroFormattingString>
			</prefix>
		</exportFileName>
		<idMapId>MyIdMap</idMapId>
		<forecastSelectionPeriod end="0" unit="day"/>
	</general>
"""

xml_validation_input = XMLValidationInput(
    xml_content=xml_snippet, schema_option=FEWSSchemaOption.TIME_SERIES_EXPORT_RUN
)
xml_validation_output: XMLValidationOutput = await validate_xml(
    ctx=ctx, input=xml_validation_input
)

print(format_as_xml(xml_validation_output))

<is_valid>False</is_valid>
<errors>
  <item>
    <reason>Validation failed: {'end': '0', 'unit': 'day'} with XsdAttributeGroup(['start', 'end', 'startOverrulable', 'endOverrulable', 'unit']):

Reason: missing required attribute 'start'</reason>
    <xpath>Found indirect parent 'timeSeriesExportRun' for 'general'. Child 'export' can contain 'general'</xpath>
  </item>
</errors>
<warnings />
<schema_location>data/fews_schemas/fewsdocs_deltares_nl/timeSeriesExportRun.xsd</schema_location>


### XML Templating

In [6]:
# partial xml template generation

from talk_with_fews.tools.validator.generate_xml_template import generate_xml_template
from talk_with_fews.models.generate_xml_template import (
    XMLTemplateInput,
    XMLTemplateOutput,
)
from talk_with_fews.models.fews_schema_options import FEWSSchemaOption

xml_template_input = XMLTemplateInput(
    schema_option=FEWSSchemaOption.TIME_SERIES_IMPORT_RUN, target_element="general"
)
xml_template_output: XMLTemplateOutput = await generate_xml_template(
    ctx=ctx, input=xml_template_input
)

print(xml_template_output.template)

<general>
  <importTypeStandard>example_value</importTypeStandard>
  <importType>example_value</importType>
  <parserClassName>example_value</parserClassName>
  <folder>example_value</folder>
  <fileNameForecastCreationDateTimePattern>example_value</fileNameForecastCreationDateTimePattern>
  <fileNameForecastCreationDateTimePatternFromZip>example_value</fileNameForecastCreationDateTimePatternFromZip>
  <jdbcDriverClass>example_value</jdbcDriverClass>
  <jdbcConnectionString>example_value</jdbcConnectionString>
  <serverUrl>example_value</serverUrl>
  <user>example_value</user>
  <encryptedPassword>example_value</encryptedPassword>
  <password>example_value</password>
  <oauth2Config>
    <authUrl>example_value</authUrl>
  </oauth2Config>
  <relativeViewPeriod/>
  <startDateTime/>
  <endDateTime/>
  <forecastSearchPeriod/>
  <externalForecastTimesSearchRelativePeriod/>
  <externalForecastTimesCardinalTimeStep/>
  <logErrorsAsWarnings>true</logErrorsAsWarnings>
  <logErrorsAsWarningsToFi

In [7]:
# full xml template generation
xml_template_input = XMLTemplateInput(
    schema_option=FEWSSchemaOption.TIME_SERIES_IMPORT_RUN, include_optional=True
)
xml_template_output: XMLTemplateOutput = await generate_xml_template(
    ctx=ctx, input=xml_template_input
)

In [8]:
print(xml_template_output.template)

<timeSeriesImportRun xmlns="http://www.wldelft.nl/fews">
<timeSeriesImportRun>
  <import>
    <general>
      <description>example_value</description>
      <importTypeStandard>example_value</importTypeStandard>
      <importType>example_value</importType>
      <parserClassName>example_value</parserClassName>
      <binDir>example_value</binDir>
      <moduleDataSetName>example_value</moduleDataSetName>
      <folder>example_value</folder>
      <fileNameDateTimeFilter>
        <timeStep>
          <description>example_value</description>
        </timeStep>
        <dateTimePattern>example_value</dateTimePattern>
        <preFixLength>1</preFixLength>
        <postFixLength>1</postFixLength>
      </fileNameDateTimeFilter>
      <fileNamePatternFilter>example_value</fileNamePatternFilter>
      <fileNameEnsembleMemberIndexPattern>example_value</fileNameEnsembleMemberIndexPattern>
      <fileNameObservationDateTimePattern>example_value</fileNameObservationDateTimePattern>
      <fileN